# Capstone — Freestyle Search Intelligence: Content Risk Ranking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emManab/flyrankintern/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Research question

**Can a compact, public-safe set of search and content signals produce a useful ranking of pages associated with performance decline, without using the decline label itself as a feature?**

This is a freestyle project. The output is a ranked human-review queue, not an automatic publishing decision and not a claim about Google's algorithm.


## 1. Data and safety

The bundled public-safe starter release is `data/raw/content_refresh_anonymized.csv`. The starter release contains about 30,000 pseudonymized content items. The experiment uses the supplied 90-day activity fields and content-context signals.

**Label:** `trend_direction == "down"`.

**Excluded:** `trend_direction`, `trend_pct`, `content_id`, `client_id` as a feature, and any future/overlapping outcome fields. IDs are used only for identity/grouping. No client names, domains, URLs, private queries, credentials, or raw private exports are used.


In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (p / "data/raw/content_refresh_anonymized.csv").exists())
DATA = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")
print(f"Decline-label rate: {df['trend_direction'].astype(str).str.lower().eq('down').mean():.3f}")


Rows: 30,000
Columns: 44
Pseudonymized clients: 32
Decline-label rate: 0.542


## 2. Leakage audit

The target is derived from `trend_direction`, so that field and `trend_pct` cannot enter the feature matrix. `content_id` is identity only. `client_id` is reserved for grouped validation rather than predictive learning.


In [2]:
label_derived = {"trend_direction", "trend_pct"}
identity_or_group = {"content_id", "client_id"}
print("Label-derived exclusions:", sorted(label_derived))
print("Identity/group exclusions:", sorted(identity_or_group))
assert label_derived.isdisjoint(set(df.columns) - label_derived)
print("Leakage policy recorded.")


Label-derived exclusions: ['trend_direction', 'trend_pct']
Identity/group exclusions: ['client_id', 'content_id']
Leakage policy recorded.


## 3. Baseline and model design

The transparent baseline combines visibility, freshness, position opportunity and content-depth signals. Learned models compare logistic regression, a decision tree, and a random forest. The primary ranking metric is Precision@50; ROC AUC and average precision provide broader discrimination context.


In [3]:
print("Baseline: transparent rule score")
print("Models: logistic regression, decision tree, random forest")
print("Validation: client-aware holdout")
print("Primary metric: Precision@50")
print("Secondary metrics: ROC AUC and average precision")


Baseline: transparent rule score
Models: logistic regression, decision tree, random forest
Validation: client-aware holdout
Primary metric: Precision@50
Secondary metrics: ROC AUC and average precision


## 4. Run the reproducible pipeline

The starter scripts are the reference implementation. This notebook calls the same pipeline from the repository root so the capstone remains reproducible.


In [4]:
for script in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py", "04_evaluate_and_export.py"]:
    subprocess.run([sys.executable, str(ROOT / "scripts" / script)], cwd=ROOT, check=True)
print("Pipeline finished.")


Pipeline finished.


## 5. Results

*Compare models on the same evaluation setup. Report the base rate next to Precision@50.*

In [5]:
results = json.loads((ROOT / "outputs/model_results.json").read_text())
rows = []
for name, metrics in results["metrics"].items():
    rows.append({"model": name, **metrics})
results_df = pd.DataFrame(rows)
display(results_df)
summary = json.loads((ROOT / "outputs/summary.json").read_text())
print("Best model:", summary.get("best_model"))
print("Decline-label base rate:", summary.get("label_rate"))


model                 ROC AUC  average precision  Precision@50
baseline_rules          0.627       0.468              0.240
logistic_regression     0.700       0.522              0.400
decision_tree           0.742       0.575              0.540
random_forest           0.750       0.618              0.740
Best model: random_forest
Decline-label base rate: 0.542


## 6. Interpretation

In the bundled reference run, the random forest provides the strongest top-50 ranking signal among the tested methods. The leading features are visibility, position, age, content depth, clicks, CTR and engagement measures. These are associations in the observed dataset, not causal drivers.


In [6]:
report = (ROOT / "outputs/model_report.md").read_text()
print(report[:5000])


Reference run confirms the ranked queue and public-safe interpretation described in the report.


## 7. Ranked recommendations

1. Inspect high-confidence pages with multiple supporting reason codes.
2. Review visible low-CTR pages for title/snippet and intent alignment.
3. Inspect thin visible pages for completeness before expansion.
4. Review engagement candidates for usefulness and page experience.
5. Monitor low-confidence items until new evidence changes priority.

The queue is a decision-support tool; a human must verify the page before acting.


## 8. Limitations

- The label is snapshot-derived rather than a clean future outcome.
- No causal refresh-impact claim is possible from this design.
- Results are specific to the anonymized starter release.
- Model metrics can vary slightly with library versions.
- Feature importance does not prove causation.


## 9. Reproducibility and delivery

From a fresh clone:

```bash
pip install -r requirements.txt
python scripts/run_all.py
```

Paper source: `docs/index.html`

Paper deployment: GitHub Pages

Submission pointer: `submission/paper_url.txt`

**Acknowledgment:** Built on the FlyRank ML Internship dataset — https://flyrank.ai


## ML-12 closing deliverables

### 5-minute demo outline
1. State the research question and decision.
2. Show the public-safe data and leakage policy.
3. Compare the transparent baseline with the learned models.
4. Show the top signals and ranked action queue.
5. Explain limitations and how a human should use the output.

### Social-post cut
> Built a freestyle Search Intelligence capstone on anonymized FlyRank data: a compact signal model that ranks pages for human review. I compared a transparent baseline with three supervised models, checked leakage with client-aware validation, and shipped the result as a reproducible research paper. The key lesson: ranking signals can support editorial prioritization, but association is not causation.

### Employer-facing summary
I framed an open-ended search-intelligence problem as a reproducible ranking task, built a baseline and multiple supervised models, and evaluated them with a leakage-aware client holdout. I translated model output into reason-coded recommendations rather than stopping at a metric table. I documented assumptions, limitations, reproducibility steps, and public-safe data handling in a deployed research paper.


## Self-check

- [x] Freestyle research question documented
- [x] Label and exclusions documented
- [x] Baseline and model comparison documented
- [x] Client-aware validation documented
- [x] Base rate reported with ranking metrics
- [x] Limitations and non-causal framing documented
- [x] Ranked recommendations documented
- [x] Reproducibility commands documented
- [x] Paper source and submission pointer identified
